# 🎨 Comic Studio — Full Backend (FLUX + PuLID)

## Every session: run A1 → A2 → A3 → A4 → A5 → A6
## First time only: also run B1 → B2 → B3 → B4 (takes ~45 min, then never again)

```
Your PC (localhost:3000)  ──ngrok──►  FastAPI (Colab)  ──►  ComfyUI (Colab)
                                            │                      │
                                       Google Drive ◄─────── FLUX models, LoRAs, images, DB
```

---
# ━━ SECTION A — Run Every Session ━━

In [ ]:
# A1 — Mount Drive & set all paths
from google.colab import drive
drive.mount('/content/drive')
import os

DRIVE_ROOT   = '/content/drive/MyDrive/ComicStudio'
MODELS_DIR   = f'{DRIVE_ROOT}/models'
LORAS_DIR    = f'{DRIVE_ROOT}/loras'
STORAGE_PATH = f'{DRIVE_ROOT}/storage'
DB_PATH      = f'{DRIVE_ROOT}/comic_studio.db'
BACKEND_SRC  = f'{DRIVE_ROOT}/backend'
BACKEND_WORK = '/content/comic_studio'
COMFYUI_DIR  = '/content/ComfyUI'

for d in [DRIVE_ROOT, MODELS_DIR, LORAS_DIR,
          # FLUX model directories
          f'{MODELS_DIR}/unet',         f'{MODELS_DIR}/vae',
          f'{MODELS_DIR}/clip',         f'{MODELS_DIR}/pulid',
          # Shared / SDXL rollback directories
          f'{MODELS_DIR}/ipadapter',    f'{MODELS_DIR}/clip_vision',
          f'{MODELS_DIR}/checkpoints',  f'{MODELS_DIR}/loras',
          f'{MODELS_DIR}/controlnet',   f'{MODELS_DIR}/instantid',
          f'{MODELS_DIR}/insightface/models',
          STORAGE_PATH,
          f'{STORAGE_PATH}/characters', f'{STORAGE_PATH}/variations',
          f'{STORAGE_PATH}/sheets',     f'{STORAGE_PATH}/panels',
          f'{STORAGE_PATH}/exports',    f'{STORAGE_PATH}/temp']:
    os.makedirs(d, exist_ok=True)

print('✅ A1 done — Drive mounted')
print(f'   Models  → {MODELS_DIR}')
print(f'   LoRAs   → {LORAS_DIR}')
print(f'   Storage → {STORAGE_PATH}')

In [ ]:
# A2 — Sync backend code from Drive → Colab disk
import shutil, os

if not os.path.exists(BACKEND_SRC):
    raise RuntimeError(f'Backend not on Drive! Upload your backend/ folder to {BACKEND_SRC}')

if os.path.exists(BACKEND_WORK):
    shutil.rmtree(BACKEND_WORK)
os.makedirs(BACKEND_WORK, exist_ok=True)
shutil.copytree(
    BACKEND_SRC, f'{BACKEND_WORK}/backend',
    ignore=shutil.ignore_patterns('venv', '__pycache__', '*.pyc', '*.pyo')
)
print(f'✅ A2 done — backend synced')
print(f'   Files: {os.listdir(BACKEND_WORK)}')

In [ ]:
# A3 — Install Python dependencies
%cd /content/comic_studio
!pip install -q fastapi uvicorn[standard] sqlalchemy pydantic pydantic-settings \
    python-multipart Pillow httpx pydrive2 python-dotenv aiofiles pyngrok \
    insightface onnxruntime-gpu huggingface_hub
print('✅ A3 done — dependencies installed')

In [ ]:
# A4 — Symlink Drive models into ComfyUI (fast, no copy)
import os

if not os.path.exists(COMFYUI_DIR):
    raise RuntimeError('ComfyUI not installed! Run Section B first.')

# Ensure all ComfyUI model dirs exist before symlinking
for d in ['unet', 'vae', 'clip', 'pulid', 'ipadapter', 'clip_vision',
          'checkpoints', 'controlnet', 'instantid', 'insightface']:
    os.makedirs(f'{COMFYUI_DIR}/models/{d}', exist_ok=True)

links = [
    # ── FLUX models ──────────────────────────────────────────────────────
    (f'{MODELS_DIR}/unet',        f'{COMFYUI_DIR}/models/unet'),
    (f'{MODELS_DIR}/vae',         f'{COMFYUI_DIR}/models/vae'),
    (f'{MODELS_DIR}/clip',        f'{COMFYUI_DIR}/models/clip'),
    (f'{MODELS_DIR}/pulid',       f'{COMFYUI_DIR}/models/pulid'),
    # ── Shared (IP-Adapter used by both FLUX and SDXL rollback) ─────────
    (f'{MODELS_DIR}/ipadapter',   f'{COMFYUI_DIR}/models/ipadapter'),
    (f'{MODELS_DIR}/clip_vision', f'{COMFYUI_DIR}/models/clip_vision'),
    (LORAS_DIR,                   f'{COMFYUI_DIR}/models/loras'),
    # ── SDXL rollback models ─────────────────────────────────────────────
    (f'{MODELS_DIR}/checkpoints', f'{COMFYUI_DIR}/models/checkpoints'),
    (f'{MODELS_DIR}/controlnet',  f'{COMFYUI_DIR}/models/controlnet'),
    (f'{MODELS_DIR}/instantid',   f'{COMFYUI_DIR}/models/instantid'),
    (f'{MODELS_DIR}/insightface', f'{COMFYUI_DIR}/models/insightface'),
]
for src, dst in links:
    if os.path.islink(dst): os.unlink(dst)
    elif os.path.exists(dst): import shutil; shutil.rmtree(dst)
    os.symlink(src, dst)
    print(f'   🔗 {os.path.basename(dst)}')

print('✅ A4 done — models symlinked')

In [ ]:
# A5 — Configure environment (FLUX)
# ═══════════════════════════════════════════════════════════════════
HF_TOKEN           = 'paste_your_hf_token_here'  # huggingface.co/settings/tokens
STYLE_LORA_TRIGGER = ''   # e.g. 'comicstyle' — leave blank if no style LoRA yet
STYLE_LORA_WEIGHT  = '1.0'
# ═══════════════════════════════════════════════════════════════════

import os
STYLE_LORA_PATH = ''

env = f'''DATABASE_URL=sqlite:///{DB_PATH}
STORAGE_PATH={STORAGE_PATH}
GENERATION_BACKEND=comfyui
COMFYUI_URL=http://127.0.0.1:8188
COMFYUI_OUTPUT_DIR=/content/ComfyUI/output
# ── FLUX model filenames (must match files in Drive/models/) ──
FLUX_UNET=flux1-dev.safetensors
FLUX_VAE=ae.safetensors
FLUX_T5=t5xxl_fp8_e4m3fn.safetensors
FLUX_CLIP_L=clip_l.safetensors
PULID_MODEL=pulid_flux_v0.9.1.safetensors
FLUX_IPADAPTER=ip-adapter.bin
# ── Optional style LoRA ────────────────────────────────────────
STYLE_LORA_PATH={STYLE_LORA_PATH}
STYLE_LORA_TRIGGER={STYLE_LORA_TRIGGER}
STYLE_LORA_WEIGHT={STYLE_LORA_WEIGHT}
# ── Background removal (false = let FLUX handle white bg natively)
REMOVE_BACKGROUND_ENABLED=false
GDRIVE_SYNC_ENABLED=false
'''
with open(f'{BACKEND_WORK}/backend/.env', 'w') as f:
    f.write(env)

# Set HF token for B4 downloads (gated FLUX model)
os.environ['HF_TOKEN'] = HF_TOKEN

loras = [x for x in os.listdir(LORAS_DIR) if x.endswith('.safetensors')]
print('✅ A5 done — FLUX env configured')
print(f'   FLUX UNET   : flux1-dev.safetensors')
print(f'   PuLID model : pulid_flux_v0.9.1.safetensors')
print(f'   Style LoRA  : {STYLE_LORA_TRIGGER or "none"}')
print(f'   All LoRAs   : {loras or ["none yet"]}')

In [ ]:
# A5b — (Optional) Link a character LoRA to a character in the DB
# Edit and run this anytime you train a new character LoRA
from sqlalchemy import create_engine, text
import os

CHARACTER_NAME    = 'Clara'                  # must match name in UI
CHAR_LORA_FILE    = 'clara_v1.safetensors'   # file in ComicStudio/loras/
CHAR_LORA_TRIGGER = 'clara_v1'
CHAR_LORA_WEIGHT  = 0.85

lora_path = f'{LORAS_DIR}/{CHAR_LORA_FILE}'
if not os.path.exists(lora_path):
    print(f'❌ File not found: {lora_path}  — upload it to Drive first')
else:
    engine = create_engine(f'sqlite:///{DB_PATH}', connect_args={'check_same_thread': False})
    with engine.connect() as conn:
        row = conn.execute(text('SELECT id FROM characters WHERE name LIKE :n'), {'n': CHARACTER_NAME}).fetchone()
        if not row:
            print(f'❌ Character "{CHARACTER_NAME}" not in DB')
            print('Available:', [r[0] for r in conn.execute(text('SELECT name FROM characters')).fetchall()])
        else:
            conn.execute(text('UPDATE characters SET lora_path=:p,lora_trigger_word=:t,lora_weight=:w WHERE id=:id'),
                         {'p': lora_path, 't': CHAR_LORA_TRIGGER, 'w': CHAR_LORA_WEIGHT, 'id': row[0]})
            conn.commit()
            print(f'✅ LoRA linked → {CHARACTER_NAME} will use ({CHAR_LORA_TRIGGER}:{CHAR_LORA_WEIGHT}) in every panel')

In [ ]:
# A6 — Start ComfyUI + FastAPI + ngrok  (keep this cell running!)
import subprocess, threading, time, os, requests
from pyngrok import ngrok

# ═══════════════════════════════════════════════════
NGROK_AUTH_TOKEN = 'paste_your_token_here'
# ═══════════════════════════════════════════════════

ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# ── Start ComfyUI ──
def run_comfyui():
    subprocess.run(['python', f'{COMFYUI_DIR}/main.py',
                    '--listen', '0.0.0.0', '--port', '8188', '--disable-auto-launch'],
                   cwd=COMFYUI_DIR)
threading.Thread(target=run_comfyui, daemon=True).start()

# Wait for ComfyUI to load the model into VRAM (can take 30-60s)
print('⏳ Waiting for ComfyUI to load...')
for i in range(60):
    try:
        r = requests.get('http://localhost:8188/system_stats', timeout=3)
        if r.status_code == 200:
            print(f'✅ ComfyUI ready — {r.json().get("system",{}).get("comfyui_version","ok")}')
            break
    except:
        pass
    time.sleep(2)
    print(f'   ... {(i+1)*2}s', end='\r')
else:
    print('⚠️  ComfyUI slow to start — check logs above')

# ── Start FastAPI ──
def run_api():
    subprocess.run(['python', '-m', 'uvicorn', 'backend.main:app',
                    '--host', '0.0.0.0', '--port', '8000'], cwd=BACKEND_WORK)
threading.Thread(target=run_api, daemon=True).start()
time.sleep(4)

# ── Open ngrok tunnel ──
tunnel = ngrok.connect(8000)
PUBLIC_URL = tunnel.public_url

# Verify FastAPI
try:
    r = requests.get(f'http://localhost:8000/health', timeout=5)
    print(f'✅ FastAPI ready — {r.json()}')
except:
    print('⚠️  FastAPI not responding — check for import errors above')

print()
print('=' * 60)
print('🎨  Comic Studio is LIVE')
print('=' * 60)
print(f'''
  Public URL  : {PUBLIC_URL}
  API Docs    : {PUBLIC_URL}/docs
  ComfyUI     : http://localhost:8188  (internal)

  On your PC:
  1. Open frontend/.env.local
  2. Set: NEXT_PUBLIC_API_URL={PUBLIC_URL}
  3. Run: npm run dev
  4. Open: http://localhost:3000
''')

while True:
    time.sleep(300)
    print(f'♥ alive — {PUBLIC_URL}')

---
# ━━ SECTION B — First Time Only ━━

Run once. Models are cached on Drive. Never run again after that.

In [ ]:
# B1 — Install ComfyUI  (~2 min)
import os
if not os.path.exists(COMFYUI_DIR):
    !git clone https://github.com/comfyanonymous/ComfyUI {COMFYUI_DIR}
else:
    print('ComfyUI already installed — pulling latest')
    !git -C {COMFYUI_DIR} pull

%cd {COMFYUI_DIR}
!pip install -q -r requirements.txt
print('✅ B1 done — ComfyUI installed')

In [ ]:
# B2 — Install custom nodes: PuLID + IP-Adapter + InsightFace  (~3 min)
import os
nodes_dir = f'{COMFYUI_DIR}/custom_nodes'

nodes = [
    # PuLID for FLUX — identity without style damage
    ('PuLID_ComfyUI',          'https://github.com/cubiq/PuLID_ComfyUI.git'),
    # IP-Adapter Plus — handles both SDXL and FLUX IP-Adapter models
    ('ComfyUI_IPAdapter_plus', 'https://github.com/cubiq/ComfyUI_IPAdapter_plus.git'),
    # ControlNet aux preprocessors (kept for SDXL rollback)
    ('comfyui_controlnet_aux', 'https://github.com/Fannovel16/comfyui_controlnet_aux.git'),
    # InstantID (kept for SDXL rollback)
    ('ComfyUI_InstantID',      'https://github.com/cubiq/ComfyUI_InstantID.git'),
]
for name, url in nodes:
    dst = f'{nodes_dir}/{name}'
    if not os.path.exists(dst):
        !git clone {url} {dst}
    else:
        print(f'✅ {name} already installed — pulling latest')
        !git -C {dst} pull --quiet
    req = f'{dst}/requirements.txt'
    if os.path.exists(req):
        !pip install -q -r {req}

!pip install -q insightface onnxruntime-gpu
print('✅ B2 done — custom nodes installed')

In [ ]:
# B3 — Download SDXL + InstantID models (rollback/optional)  (~15-30 min first time)
# Skip this if you don't need SDXL rollback capability.
from huggingface_hub import hf_hub_download
import os, shutil

def get(repo, filename, dest_dir, dest_name=None, token=None):
    dest_name = dest_name or os.path.basename(filename)
    dest = f'{dest_dir}/{dest_name}'
    if os.path.exists(dest):
        mb = os.path.getsize(dest)/1024/1024
        print(f'  ✅ cached — {dest_name} ({mb:.0f} MB)')
        return dest
    print(f'  ⬇️  {dest_name}...')
    tmp = hf_hub_download(repo_id=repo, filename=filename, token=token)
    os.makedirs(dest_dir, exist_ok=True)
    shutil.copy(tmp, dest)
    mb = os.path.getsize(dest)/1024/1024
    print(f'  ✅ saved — {dest_name} ({mb:.0f} MB)')
    return dest

print('\n[1/4] SDXL Base (~6.9 GB)...')
get('stabilityai/stable-diffusion-xl-base-1.0',
    'sd_xl_base_1.0.safetensors', f'{MODELS_DIR}/checkpoints')

print('\n[2/4] InstantID IP-Adapter...')
get('InstantX/InstantID', 'ip-adapter.bin',
    f'{MODELS_DIR}/instantid', 'ip-adapter_instantid_sdxl.bin')

print('\n[3/4] InstantID ControlNet...')
get('InstantX/InstantID',
    'ControlNetModel/diffusion_pytorch_model.safetensors',
    f'{MODELS_DIR}/controlnet', 'instantid_controlnet.safetensors')

print('\n[4/4] InsightFace AntelopeV2...')
antelop_dir = f'{MODELS_DIR}/insightface/models/antelopev2'
os.makedirs(antelop_dir, exist_ok=True)
for fname in ['1k3d68.onnx', '2d106det.onnx', 'genderage.onnx',
              'glintr100.onnx', 'scrfd_10g_bnkps.onnx']:
    get('DIAMONIK7777/antelopev2', fname, antelop_dir)

print('\n✅ B3 done — SDXL rollback models cached')

In [ ]:
# B4 — Download FLUX models → cached on Drive  (~30-45 min first time)
# Requires: HF_TOKEN set in A5 with FLUX.1-dev license accepted.
# Accept license at: https://huggingface.co/black-forest-labs/FLUX.1-dev
from huggingface_hub import hf_hub_download
import os, shutil

HF = os.environ.get('HF_TOKEN', '')  # set in A5
if not HF:
    raise RuntimeError('HF_TOKEN not set — run A5 first and paste your token')

def get(repo, filename, dest_dir, dest_name=None, token=None):
    dest_name = dest_name or os.path.basename(filename)
    dest = f'{dest_dir}/{dest_name}'
    if os.path.exists(dest):
        mb = os.path.getsize(dest)/1024/1024
        print(f'  ✅ cached — {dest_name} ({mb:.0f} MB)')
        return dest
    print(f'  ⬇️  {dest_name}...')
    tmp = hf_hub_download(repo_id=repo, filename=filename, token=token)
    os.makedirs(dest_dir, exist_ok=True)
    shutil.copy(tmp, dest)
    mb = os.path.getsize(dest)/1024/1024
    print(f'  ✅ saved — {dest_name} ({mb:.0f} MB)')
    return dest

print('\n[1/6] FLUX.1-dev UNET (~23 GB) — gated, needs HF token...')
get('black-forest-labs/FLUX.1-dev', 'flux1-dev.safetensors',
    f'{MODELS_DIR}/unet', token=HF)

print('\n[2/6] FLUX VAE (~335 MB)...')
get('black-forest-labs/FLUX.1-dev', 'ae.safetensors',
    f'{MODELS_DIR}/vae', token=HF)

print('\n[3/6] T5-XXL fp8 text encoder (~5 GB)...')
get('comfyanonymous/flux_text_encoders', 't5xxl_fp8_e4m3fn.safetensors',
    f'{MODELS_DIR}/clip')

print('\n[4/6] CLIP-L text encoder (~235 MB)...')
get('comfyanonymous/flux_text_encoders', 'clip_l.safetensors',
    f'{MODELS_DIR}/clip')

print('\n[5/6] PuLID FLUX model (~1.1 GB)...')
get('guozinan/PuLID', 'pulid_flux_v0.9.1.safetensors',
    f'{MODELS_DIR}/pulid')

print('\n[6/6] IP-Adapter FLUX (~1.1 GB)...')
get('InstantX/FLUX.1-dev-IP-Adapter', 'ip-adapter.bin',
    f'{MODELS_DIR}/ipadapter')

print('\n✅ B4 done — all FLUX models cached on Drive')
print('Section B is complete. You never need to run it again.')
print()
print('Model sizes on Drive:')
for sub, name in [('unet','FLUX UNET'), ('vae','VAE'), ('clip','Text encoders'),
                  ('pulid','PuLID'), ('ipadapter','IP-Adapter')]:
    d = f'{MODELS_DIR}/{sub}'
    files = os.listdir(d) if os.path.exists(d) else []
    total_mb = sum(os.path.getsize(f'{d}/{f}')/1024/1024 for f in files if os.path.isfile(f'{d}/{f}'))
    print(f'  {name:<20}: {total_mb:.0f} MB  ({files})')

In [ ]:
# B4 — Download FLUX models → cached on Drive  (~30-45 min first time)
# Requires: HF_TOKEN set in A5 with FLUX.1-dev license accepted.
# Accept license at: https://huggingface.co/black-forest-labs/FLUX.1-dev
from huggingface_hub import hf_hub_download
import os, shutil

HF = os.environ.get('HF_TOKEN', '')  # set in A5
if not HF:
    raise RuntimeError('HF_TOKEN not set — run A5 first and paste your token')

def get(repo, filename, dest_dir, dest_name=None, token=None):
    dest_name = dest_name or os.path.basename(filename)
    dest = f'{dest_dir}/{dest_name}'
    if os.path.exists(dest):
        mb = os.path.getsize(dest)/1024/1024
        print(f'  ✅ cached — {dest_name} ({mb:.0f} MB)')
        return dest
    print(f'  ⬇️  {dest_name}...')
    tmp = hf_hub_download(repo_id=repo, filename=filename, token=token)
    os.makedirs(dest_dir, exist_ok=True)
    shutil.copy(tmp, dest)
    mb = os.path.getsize(dest)/1024/1024
    print(f'  ✅ saved — {dest_name} ({mb:.0f} MB)')
    return dest

print('\n[1/6] FLUX.1-dev UNET (~23 GB) — gated, needs HF token...')
get('black-forest-labs/FLUX.1-dev', 'flux1-dev.safetensors',
    f'{MODELS_DIR}/unet', token=HF)

print('\n[2/6] FLUX VAE (~335 MB)...')
get('black-forest-labs/FLUX.1-dev', 'ae.safetensors',
    f'{MODELS_DIR}/vae', token=HF)

print('\n[3/6] T5-XXL fp8 text encoder (~5 GB)...')
get('comfyanonymous/flux_text_encoders', 't5xxl_fp8_e4m3fn.safetensors',
    f'{MODELS_DIR}/clip')

print('\n[4/6] CLIP-L text encoder (~235 MB)...')
get('comfyanonymous/flux_text_encoders', 'clip_l.safetensors',
    f'{MODELS_DIR}/clip')

print('\n[5/6] PuLID FLUX model (~1.1 GB)...')
get('guozinan/PuLID', 'pulid_flux_v0.9.1.safetensors',
    f'{MODELS_DIR}/pulid')

print('\n[6/6] IP-Adapter FLUX (~1.1 GB)...')
get('InstantX/FLUX.1-dev-IP-Adapter', 'ip-adapter.bin',
    f'{MODELS_DIR}/ipadapter')

print('\n✅ B4 done — all FLUX models cached on Drive')
print('Section B is complete. You never need to run it again.')
print()
print('Model sizes on Drive:')
for sub, name in [('unet','FLUX UNET'), ('vae','VAE'), ('clip','Text encoders'),
                  ('pulid','PuLID'), ('ipadapter','IP-Adapter')]:
    d = f'{MODELS_DIR}/{sub}'
    files = os.listdir(d) if os.path.exists(d) else []
    total_mb = sum(os.path.getsize(f'{d}/{f}')/1024/1024 for f in files if os.path.isfile(f'{d}/{f}'))
    print(f'  {name:<20}: {total_mb:.0f} MB  ({files})')

In [ ]:
# B4 — Download FLUX models → cached on Drive  (~30-45 min first time)
# Requires: HF_TOKEN set in A5 with FLUX.1-dev license accepted.
# Accept license at: https://huggingface.co/black-forest-labs/FLUX.1-dev
from huggingface_hub import hf_hub_download
import os, shutil

HF = os.environ.get('HF_TOKEN', '')  # set in A5
if not HF:
    raise RuntimeError('HF_TOKEN not set — run A5 first and paste your token')

def get(repo, filename, dest_dir, dest_name=None, token=None):
    dest_name = dest_name or os.path.basename(filename)
    dest = f'{dest_dir}/{dest_name}'
    if os.path.exists(dest):
        mb = os.path.getsize(dest)/1024/1024
        print(f'  ✅ cached — {dest_name} ({mb:.0f} MB)')
        return dest
    print(f'  ⬇️  {dest_name}...')
    tmp = hf_hub_download(repo_id=repo, filename=filename, token=token)
    os.makedirs(dest_dir, exist_ok=True)
    shutil.copy(tmp, dest)
    mb = os.path.getsize(dest)/1024/1024
    print(f'  ✅ saved — {dest_name} ({mb:.0f} MB)')
    return dest

print('\n[1/6] FLUX.1-dev UNET (~23 GB) — gated, needs HF token...')
get('black-forest-labs/FLUX.1-dev', 'flux1-dev.safetensors',
    f'{MODELS_DIR}/unet', token=HF)

print('\n[2/6] FLUX VAE (~335 MB)...')
get('black-forest-labs/FLUX.1-dev', 'ae.safetensors',
    f'{MODELS_DIR}/vae', token=HF)

print('\n[3/6] T5-XXL fp8 text encoder (~5 GB)...')
get('comfyanonymous/flux_text_encoders', 't5xxl_fp8_e4m3fn.safetensors',
    f'{MODELS_DIR}/clip')

print('\n[4/6] CLIP-L text encoder (~235 MB)...')
get('comfyanonymous/flux_text_encoders', 'clip_l.safetensors',
    f'{MODELS_DIR}/clip')

print('\n[5/6] PuLID FLUX model (~1.1 GB)...')
get('guozinan/PuLID', 'pulid_flux_v0.9.1.safetensors',
    f'{MODELS_DIR}/pulid')

print('\n[6/6] IP-Adapter FLUX (~1.1 GB)...')
get('InstantX/FLUX.1-dev-IP-Adapter', 'ip-adapter.bin',
    f'{MODELS_DIR}/ipadapter')

print('\n✅ B4 done — all FLUX models cached on Drive')
print('Section B is complete. You never need to run it again.')
print()
print('Model sizes on Drive:')
for sub, name in [('unet','FLUX UNET'), ('vae','VAE'), ('clip','Text encoders'),
                  ('pulid','PuLID'), ('ipadapter','IP-Adapter')]:
    d = f'{MODELS_DIR}/{sub}'
    files = os.listdir(d) if os.path.exists(d) else []
    total_mb = sum(os.path.getsize(f'{d}/{f}')/1024/1024 for f in files if os.path.isfile(f'{d}/{f}'))
    print(f'  {name:<20}: {total_mb:.0f} MB  ({files})')

---
# ━━ UTILITIES ━━

Run anytime for diagnostics.

In [ ]:
# U1 — Inspect everything on Drive
from sqlalchemy import create_engine, text
import os

print('📁 Models on Drive:')
for sub in ['checkpoints', 'instantid', 'controlnet', 'ipadapter',
            'clip_vision', 'insightface/models/antelopev2']:
    d = f'{MODELS_DIR}/{sub}'
    if os.path.exists(d):
        files = [f for f in os.listdir(d) if not f.startswith('.')]
        print(f'  {sub}/: {files}')

print('\n📁 Your LoRAs:')
for f in os.listdir(LORAS_DIR):
    if f.endswith(('.safetensors', '.pt')):
        mb = os.path.getsize(f'{LORAS_DIR}/{f}')/1024/1024
        print(f'  {f}  ({mb:.0f} MB)')

print('\n📋 Characters in DB:')
try:
    engine = create_engine(f'sqlite:///{DB_PATH}', connect_args={'check_same_thread': False})
    with engine.connect() as conn:
        rows = conn.execute(text('SELECT name, lora_trigger_word, canonical_prompt FROM characters')).fetchall()
        for r in rows:
            lora = f'✅ {r[1]}' if r[1] else '❌ no LoRA'
            print(f'  {r[0]:<20} {lora}')
            if r[2]: print(f'  {" "*20} → {r[2][:70]}')
except Exception as e:
    print(f'  DB not ready yet: {e}')

In [ ]:
# U2 — Health check (run after A6 to verify everything works)
import requests

print('Checking ComfyUI...')
try:
    r = requests.get('http://localhost:8188/system_stats', timeout=5)
    info = r.json().get('system', {})
    print(f'  ✅ ComfyUI {info.get("comfyui_version","ok")} | '
          f'GPU: {info.get("gpu_name","?")} | '
          f'VRAM: {info.get("vram_free",0)/1024:.1f} GB free')
except Exception as e:
    print(f'  ❌ {e}')

print('Checking FastAPI...')
try:
    r = requests.get('http://localhost:8000/health', timeout=5)
    print(f'  ✅ {r.json()}')
    r2 = requests.get('http://localhost:8000/characters', timeout=5)
    print(f'  ✅ {len(r2.json())} characters in DB')
except Exception as e:
    print(f'  ❌ {e}')